In [2]:
import pandas as pd
import json
import os

In [3]:
df_commissions = pd.read_json("../data/model_params/commission_params.json")
df_clients = pd.read_json("../data/model_params/client_params.json")

In [4]:
def get_est_commission(prediction_type, conservative=True):
    growth_options = None
    growth_futures = None
    fee_futures = None
    fee_options = None
    if prediction_type == "value" and conservative:
        fee_futures = df_commissions['data'].iloc[6]
        fee_options = df_commissions['data'].iloc[7]
        growth_futures = df_clients['data'].iloc[0]
        growth_options = df_clients['data'].iloc[2]
    elif prediction_type == "value" and not conservative:
        fee_futures = df_commissions['data'].iloc[6]
        fee_options = df_commissions['data'].iloc[7]
        growth_futures = df_clients['data'].iloc[4]
        growth_options = df_clients['data'].iloc[6]
    elif prediction_type == "trades" and conservative:
        fee_futures = df_commissions['data'].iloc[8]
        fee_options = df_commissions['data'].iloc[9]
        growth_futures = df_clients['data'].iloc[1]
        growth_options = df_clients['data'].iloc[3]
    elif prediction_type == "trades" and not conservative:
        fee_futures = df_commissions['data'].iloc[8]
        fee_options = df_commissions['data'].iloc[9]
        growth_futures = df_clients['data'].iloc[5]
        growth_options = df_clients['data'].iloc[7]

    options_commission_growth = fee_options * growth_options
    futures_commission_growth = fee_futures * growth_futures
    return options_commission_growth, futures_commission_growth

In [5]:
# расчеты роста КД по разным показателям и разным сценариям
kd_options_value_conservative, kd_futures_value_conservative = get_est_commission("value")
kd_options_value_positive, kd_futures_value_positive = get_est_commission("value", False)
kd_options_trades_conservative, kd_futures_trades_conservative = get_est_commission("trades")
kd_options_trades_positive, kd_futures_trades_positive = get_est_commission("trades", False)

In [8]:
# сохраняем в json
kd_params = {
    "description": {
        "kd_options_value_conservative": "Options commission growth, value-based, conservative",
        "kd_futures_value_conservative": "Futures commission growth, value-based, conservative",
        "kd_options_value_positive": "Options commission growth, value-based, positive",
        "kd_futures_value_positive": "Futures commission growth, value-based, positive",
        "kd_options_trades_conservative": "Options commission growth, trades-based, conservative",
        "kd_futures_trades_conservative": "Futures commission growth, trades-based, conservative",
        "kd_options_trades_positive": "Options commission growth, trades-based, positive",
        "kd_futures_trades_positive": "Futures commission growth, trades-based, positive"
    },
    "data": {
        "kd_options_value_conservative": kd_options_value_conservative,
        "kd_futures_value_conservative": kd_futures_value_conservative,
        "kd_options_value_positive": kd_options_value_positive,
        "kd_futures_value_positive": kd_futures_value_positive,
        "kd_options_trades_conservative": kd_options_trades_conservative,
        "kd_futures_trades_conservative": kd_futures_trades_conservative,
        "kd_options_trades_positive": kd_options_trades_positive,
        "kd_futures_trades_positive": kd_futures_trades_positive
    }
}

In [9]:
os.makedirs("../data/model_params", exist_ok=True)
file_name = "kd_growth_params.json"
with open(f"../data/model_params/{file_name}", "w") as file:
    json.dump(kd_params, file, indent=4)

В json файле лежат значения того, насколько вырастет комиссионный доход при приходе одного клиента на рынок опционов.

Значения зависят от типа предсказания, но помогают понять примерные доходы биржи от внедрения инициатив с учетом прогнозируемого притока клиентов на рынок опционов.

Стоит упомянуть про потенциальный рост комиссионных доходов еще сильнее в силу частичного притока клиентов на рынок фьючерсов.